# Dense-16 HELM assumption audit

This notebook deliberately pauses new routing experiments and audits the **vanilla 16-head dense baseline**.

It tests assumptions we have been treating as facts:

- Is the dense model's head-level effective rank actually high?
- Is low raw rank caused by duplicate directions or simply unequal head strength?
- Does collapse exist **before** the output projection, or does `W_O` create/concentrate it?
- Are all 16 heads actually useful to MLM CE?
- Does residual RMS/energy predict exact head importance?
- How quickly does a model trained dense degrade when heads are removed post-hoc?
- Is the 6.5k checkpoint representative, or does rank evolve significantly over training?

The first analysis is a deep audit of one checkpoint. The second tracks rank across several checkpoints using the same validation examples.


In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


## 1. Write the exact vanilla model

This is the dense model source you supplied. The active `HELMBlock` contains no router and all attention heads contribute on every forward pass.


In [2]:
%%writefile model.py
"""HELM_32_random.

Training-time control for HELM_7c:
- same 12-layer, 32-head, d_head=64, 2048-wide attention architecture
- NO learned router
- NO easiness labels
- NO count loss
- first 8 heads are always active to match HELM_7c's permanent backbone
- remaining 24 heads are selected uniformly at random PER EXAMPLE
- each layer uses a fixed total K matched to HELM_7c's observed mean count profile

Purpose:
isolate the effect of partial/stochastic per-head forward/backward exposure from
the effect of HELM's learned identity-selection policy.

The random mask is part of the REAL forward pass, so selected heads receive normal
CE gradients and rejected heads do not.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None

from transformers import PretrainedConfig, PreTrainedModel


def justnorm(x, dim=-1, eps=1e-12):
    return x / (x.norm(p=2, dim=dim, keepdim=True) + eps)


def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x, w, b)


class HELMConfig(PretrainedConfig):
    model_type = "helm_32_random"

    def __init__(
        self,
        # General model hyperparameters
        hidden_size=1024,
        sqrt_hidden_size=32,
        max_position_embeddings=4096,
        initializer_range=0.03125,
        num_hidden_layers=12,
        num_attention_heads=32,
        d_head=64,
        rope_theta=160000,
        intermediate_size=2816,
        norm_eps=1e-12,
        hidden_act="swiglu",
        swiglu_s_init=1.0,
        base_lr=3e-4,
        min_lr=3e-5,
        weight_decay=0.0,
        bias=False,
        use_ckpt=False,

        # Tokenization / MLM metadata
        tokenizer_path="answerdotai/ModernBERT-base",
        vocab_size=50368,
        bos_token_id=50281,
        eos_token_id=50282,
        pad_token_id=50283,
        mask_token_id=50284,
        unk_token_id=50285,
        mlm_probability=0.3,
        mlm_use_span_masking=True,
        mlm_span_length=3,

        # Random-routing control.
        num_permanent_heads=8,
        random_total_heads_per_layer=None,
        random_routing_in_eval=True,
        jitter_noise=0.01,

        # nGPT attention / FFN hyperparameters
        ngpt_sqk_init_value=1.0,
        ngpt_sqk_init_scale=0.03125,
        use_exclusive_attention=True,
        ngpt_alpha_value_attn=0.05,
        ngpt_alpha_scale_attn=0.03125,
        ngpt_alpha_value_mlp=0.05,
        ngpt_alpha_scale_mlp=0.03125,
        ngpt_suv_value=1.0,
        ngpt_suv_scale=1.0,
        ngpt_sz_init_value=1.0,
        ngpt_sz_init_scale=0.03125,
        **kwargs,
    ):
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        self.num_permanent_heads = int(num_permanent_heads)
        if random_total_heads_per_layer is None:
            # Rounded HELM_7c observed mean total-head counts from the existing audit:
            # [15.5, 19.375, 18.75, 19.4375, 19.25, 19.9375,
            #  19.9375, 19.5625, 20.125, 19.5625, 20.75, 21.0625]
            random_total_heads_per_layer = [
                16, 19, 19, 19, 19, 20, 20, 20, 20, 20, 21, 21
            ]
        self.random_total_heads_per_layer = [
            int(x) for x in random_total_heads_per_layer
        ]
        self.random_routing_in_eval = bool(random_routing_in_eval)
        self.jitter_noise = float(jitter_noise)

        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale

        if self.num_attention_heads != 32:
            raise ValueError(
                f"HELM_32_random requires num_attention_heads=32, got {self.num_attention_heads}."
            )
        if self.d_head != 64:
            raise ValueError(
                f"HELM_32_random requires d_head=64, got {self.d_head}."
            )
        if self.num_attention_heads * self.d_head != 2048:
            raise ValueError("HELM_32_random attention width must be exactly 2048.")
        if self.num_permanent_heads != 8:
            raise ValueError("HELM_32_random uses 8 always-on heads to match HELM_7c.")
        if len(self.random_total_heads_per_layer) != self.num_hidden_layers:
            raise ValueError(
                "random_total_heads_per_layer must have one value per layer"
            )
        for k in self.random_total_heads_per_layer:
            if not (self.num_permanent_heads <= k <= self.num_attention_heads):
                raise ValueError(f"Invalid random total-head count: {k}")

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id,
        )

    def forward(self, input_ids):
        return justnorm(self.word_embeddings(input_ids))


class RotaryEmbeddings(nn.Module):
    def __init__(self, dim, max_position_embeddings, rope_theta=160000):
        super().__init__()
        inv_freq = 1.0 / (
            rope_theta ** (torch.arange(0, dim, 2).float() / dim)
        )
        t = torch.arange(max_position_embeddings, dtype=inv_freq.dtype)
        freqs = torch.outer(t, inv_freq)
        freqs = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x):
        seq_len = x.shape[-2]
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)
        return (x * x_cos) + (self.rotate_half(x) * x_sin)


class HELMSelfAttention(nn.Module):
    """32-head attention with a real random per-example hard mask."""

    def __init__(self, config, layer_idx):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.num_elastic_heads = self.num_attention_heads - self.num_permanent_heads
        self.layer_idx = int(layer_idx)
        self.random_total_heads = int(
            config.random_total_heads_per_layer[self.layer_idx]
        )
        self.random_elastic_heads = (
            self.random_total_heads - self.num_permanent_heads
        )
        self.d_head = config.d_head
        self.routing_mode = "random"
        self.save_random_mask = None
        self.jitter_noise = float(config.jitter_noise)
        self.total_head_dim = self.num_attention_heads * self.d_head
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        # 1024 -> Q/K/V, each 2048 wide.
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias=config.bias,
        )

        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta,
        )

        self.sqk = nn.Parameter(
            self.ngpt_sqk_init_scale * torch.ones(self.total_head_dim)
        )

        # 2048 -> 1024, matching HELM_7c.
        self.output = nn.Linear(
            self.total_head_dim,
            config.hidden_size,
            bias=config.bias,
        )

    def set_routing_mode(self, mode="random"):
        if mode not in {"random", "dense"}:
            raise ValueError("mode must be 'random' or 'dense'")
        self.routing_mode = mode

    def _random_full_mask(self, batch_size, device, dtype):
        """Exactly K heads per example: 8 always-on + random elastic identities."""
        k = int(self.random_elastic_heads)

        if k <= 0:
            elastic = torch.zeros(
                batch_size, self.num_elastic_heads, device=device, dtype=dtype
            )
        elif k >= self.num_elastic_heads:
            elastic = torch.ones(
                batch_size, self.num_elastic_heads, device=device, dtype=dtype
            )
        else:
            # Static-shaped random Top-K. Scores are NOT parameters and carry no gradient.
            scores = torch.rand(
                batch_size, self.num_elastic_heads, device=device, dtype=torch.float32
            )
            indices = torch.topk(scores, k=k, dim=-1, largest=True, sorted=False).indices
            elastic = torch.zeros(
                batch_size, self.num_elastic_heads, device=device, dtype=dtype
            )
            elastic.scatter_(1, indices, 1.0)

        permanent = torch.ones(
            batch_size, self.num_permanent_heads, device=device, dtype=dtype
        )
        return torch.cat([permanent, elastic], dim=-1)

    def forward(self, hidden_states, attention_mask):
        batch_size, seq_len, _ = hidden_states.shape

        qkv_proj = cast_linear(hidden_states, self.qkv)
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        q = q.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)
        k = k.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)
        v = v.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)

        # Keep HELM/nGPT mechanics unchanged.
        q = justnorm(q)
        k = justnorm(k)
        q = self.RoPE(q)
        k = self.RoPE(k)

        sqk = self.sqk * (
            self.ngpt_sqk_init_value / self.ngpt_sqk_init_scale
        )
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)
        q = sqk.to(q.dtype) * q
        k = sqk.to(k.dtype) * k

        context_layer = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask.to(q.dtype),
            scale=math.sqrt(self.d_head),
        )

        if self.config.use_exclusive_attention:
            vn = F.normalize(v, dim=-1)
            context_layer = (
                context_layer
                - (context_layer * vn).sum(dim=-1, keepdim=True) * vn
            )

        use_random = self.training or (
            self.config.random_routing_in_eval and self.routing_mode == "random"
        )
        if self.routing_mode == "dense":
            use_random = False

        if use_random:
            full_mask = self._random_full_mask(
                batch_size, context_layer.device, context_layer.dtype
            )
        else:
            full_mask = torch.ones(
                batch_size,
                self.num_attention_heads,
                device=context_layer.device,
                dtype=context_layer.dtype,
            )

        # REAL hard forward masking. Rejected heads receive no CE gradient.
        context_layer = context_layer * full_mask.view(
            batch_size, self.num_attention_heads, 1, 1
        )
        self.save_random_mask = full_mask.detach()

        # Match HELM_7c's permanent-head training jitter so the control differs
        # from 7c primarily in LEARNED identity selection vs RANDOM identity selection.
        if self.training and self.num_permanent_heads > 0:
            permanent_heads = context_layer[:, :self.num_permanent_heads, :, :]
            elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]
            permanent_heads = F.dropout(
                permanent_heads,
                p=self.jitter_noise,
                training=True,
            )
            context_layer = torch.cat(
                [permanent_heads, elastic_heads], dim=1
            )

        context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()
        context_reshaped = context_reshaped.view(batch_size, seq_len, self.total_head_dim)
        return cast_linear(context_reshaped, self.output)


class HELMMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        self.attn_alpha = nn.Parameter(
            self.ngpt_alpha_scale_attn * torch.ones(self.hidden_size)
        )
        self.mlp_alpha = nn.Parameter(
            self.ngpt_alpha_scale_mlp * torch.ones(self.hidden_size)
        )

        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias=config.bias,
        )
        self.suv = nn.Parameter(
            self.ngpt_suv_scale * torch.ones(2 * self.intermediate_size)
        )
        self.silu = nn.SiLU()
        self.mlp_proj = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias,
        )

    def forward(self, hidden_states, hidden_states_attention):
        # nGPT attention residual update.
        a_norm = justnorm(hidden_states)
        b_norm = justnorm(hidden_states_attention)
        lr = self.attn_alpha * (
            self.ngpt_alpha_value_attn / self.ngpt_alpha_scale_attn
        )
        lr = torch.abs(lr).to(a_norm.dtype)
        hidden_states_opt1 = justnorm(a_norm + lr * (b_norm - a_norm))

        # nGPT SwiGLU FFN.
        uv_pre = cast_linear(hidden_states_opt1, self.mlp_exp)
        suv = self.suv * (
            self.ngpt_suv_value / self.ngpt_suv_scale
        ) * (self.hidden_size ** 0.5)
        uv_post_suv = suv.to(uv_pre.dtype) * uv_pre
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)
        x_mlp = u * self.silu(v)
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # nGPT MLP residual update.
        a_norm = justnorm(hidden_states_opt1)
        b_norm = justnorm(h_mlp)
        lr = self.mlp_alpha * (
            self.ngpt_alpha_value_mlp / self.ngpt_alpha_scale_mlp
        )
        lr = torch.abs(lr).to(a_norm.dtype)
        return justnorm(a_norm + lr * (b_norm - a_norm))


class HELMBlock(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.attn = HELMSelfAttention(config, layer_idx=layer_idx)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask):
        attn_output = self.attn(hidden_states, attention_mask)
        return self.mlp(hidden_states, attn_output)


class HELMModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList(
            [
                HELMBlock(config, layer_idx=i)
                for i in range(config.num_hidden_layers)
            ]
        )

    def forward(self, input_ids, attention_mask):
        # Additive SDPA mask: [B,S] -> [B,1,1,S].
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(
            attention_mask == 0, float("-inf")
        )
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)

        for block in self.blocks:
            if self.use_ckpt and self.training:
                ckpt_fn = (
                    _xla_checkpoint
                    if (
                        _xla_checkpoint is not None
                        and hidden_states.device.type == "xla"
                    )
                    else torch.utils.checkpoint.checkpoint
                )
                hidden_states = ckpt_fn(
                    block,
                    hidden_states,
                    attention_mask,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states = block(hidden_states, attention_mask)

        return hidden_states


class HELMForMaskedLM(PreTrainedModel):
    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=config.bias,
        )
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.initializer_range,
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.initializer_range,
            )

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # Every matrix here existed in the dense-16 baseline as well.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    def set_routing_mode(self, mode="random"):
        """Use 'random' for the matched-exposure control or 'dense' to inspect the trained bank."""
        for block in self.model.blocks:
            block.attn.set_routing_mode(mode)
        return self

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}
        for i, block in enumerate(self.model.blocks):
            attn = block.attn
            mlp = block.mlp

            sqk = attn.sqk.detach().float().cpu()
            telemetry[f"layer_{i}_sqk_mean"] = sqk.mean().item()
            telemetry[f"layer_{i}_random_total_heads"] = float(attn.random_total_heads)
            telemetry[f"layer_{i}_random_elastic_heads"] = float(attn.random_elastic_heads)
            if attn.save_random_mask is not None:
                mask = attn.save_random_mask.detach().float().cpu()
                telemetry[f"layer_{i}_actual_head_count_mean"] = mask.sum(dim=-1).mean().item()
                telemetry[f"layer_{i}_elastic_activation_frequency_hist"] = (
                    mask[:, self.config.num_permanent_heads:].mean(dim=0)
                )
            telemetry[f"layer_{i}_sqk_std"] = sqk.std(unbiased=False).item()
            telemetry[f"layer_{i}_sqk_hist"] = sqk

            attn_alpha = mlp.attn_alpha.detach().float().cpu()
            telemetry[f"layer_{i}_attn_alpha_mean"] = attn_alpha.mean().item()
            telemetry[f"layer_{i}_attn_alpha_std"] = attn_alpha.std(unbiased=False).item()
            telemetry[f"layer_{i}_attn_alpha_hist"] = attn_alpha

            mlp_alpha = mlp.mlp_alpha.detach().float().cpu()
            telemetry[f"layer_{i}_mlp_alpha_mean"] = mlp_alpha.mean().item()
            telemetry[f"layer_{i}_mlp_alpha_std"] = mlp_alpha.std(unbiased=False).item()
            telemetry[f"layer_{i}_mlp_alpha_hist"] = mlp_alpha

            suv = mlp.suv.detach().float().cpu()
            telemetry[f"layer_{i}_suv_mean"] = suv.mean().item()
            telemetry[f"layer_{i}_suv_std"] = suv.std(unbiased=False).item()
            telemetry[f"layer_{i}_suv_hist"] = suv

        sz = self.sz.detach().float().cpu()
        telemetry["lm_head_sz_mean"] = sz.mean().item()
        telemetry["lm_head_sz_std"] = sz.std(unbiased=False).item()
        telemetry["lm_head_sz_hist"] = sz
        return telemetry

    def forward(
        self,
        input_ids,
        attention_mask,
        current_step=None,
        easiness_score=None,
    ):
        # current_step/easiness_score are accepted for generic trainer compatibility.
        # Random routing does not use either value.
        features = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        sz = self.sz * (
            self.ngpt_sz_init_value / self.ngpt_sz_init_scale
        )
        unscaled_logits = cast_linear(features, self.classifier)
        return sz.to(unscaled_logits.dtype) * unscaled_logits

Writing model.py


## 2. Write the single-checkpoint audit

Key distinction: the script reports both **raw effective rank** (magnitude-sensitive) and **directional effective rank** (each head normalized first). This prevents us from calling unequal-strength but distinct heads "redundant."


In [3]:
%%writefile analyze_helm_32_random.py
"""
HELM_32_random analysis.

Primary question:
Does partial/stochastic head exposure damage the trained 32-head bank even when
there is no learned router?

Key outputs:
- random-sparse CE over multiple independent mask draws
- SAME CHECKPOINT forced-dense CE
- permanent-only CE
- potential vs executed raw rank / energy concentration
- actual random activation-frequency uniformity
- A_h/I_h specialization NEGATIVE CONTROL

For Random-32, A_h/I_h assignment is random wrt input. Therefore the expected
common-coalition A-I gap is approximately zero. This validates the specialization
test itself.
"""

from pathlib import Path
import argparse
import json
import numpy as np

from helm_analysis_core import *


MODEL_REPO = "JamesResearch1216/HELM_32_random"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--device", default="auto", choices=["auto","cuda","xla","cpu"])
    ap.add_argument("--model-file", default="model.py")
    ap.add_argument("--repo", default=MODEL_REPO)
    ap.add_argument("--checkpoint", default=CHECKPOINT_FILE)
    ap.add_argument("--checkpoint-path", default=None)
    ap.add_argument("--validation-path", default=None)
    ap.add_argument("--data-repo", default=DATA_REPO)
    ap.add_argument("--validation-file", default=VALIDATION_FILE)
    ap.add_argument("--num-examples", type=int, default=64)
    ap.add_argument("--batch-size", type=int, default=2)
    ap.add_argument("--seq-len", type=int, default=1024)
    ap.add_argument("--layers", default="0,1,2,3,4,5,6,7,8,9,10,11")
    ap.add_argument("--geometry-batches", type=int, default=4)
    ap.add_argument("--sample-tokens", type=int, default=64)
    ap.add_argument("--random-trials", type=int, default=8)
    ap.add_argument("--specialization-layers", default="0,5,11")
    ap.add_argument("--specialization-group-size", type=int, default=8)
    ap.add_argument("--ablation-batch-size", type=int, default=2)
    ap.add_argument("--skip-specialization", action="store_true")
    ap.add_argument("--seed", type=int, default=1216)
    ap.add_argument("--output-dir", default="helm_32_random_analysis")
    ap.add_argument("--cache-dir", default=".helm_32_random_analysis_cache")
    args = ap.parse_args()

    seed_everything(args.seed)
    out = Path(args.output_dir)
    cache = Path(args.cache_dir)
    out.mkdir(parents=True, exist_ok=True)
    cache.mkdir(parents=True, exist_ok=True)

    dev = resolve_device(args.device)
    token = get_hf_token()
    module = import_model_file(Path(args.model_file), "random32_arch")
    ckpt = resolve_checkpoint(
        args.checkpoint_path, args.repo, args.checkpoint, cache, token
    )
    validation = resolve_validation(
        args.validation_path,
        args.data_repo,
        args.validation_file,
        cache,
        token,
    )

    model, cfg = load_model(module, ckpt, dev, None)
    model.set_routing_mode("random")

    examples = prepare_examples(
        validation, cfg, args.num_examples, args.seq_len, args.seed
    )
    examples, batches = make_batches(examples, args.batch_size)

    print("\n=== Capture one random-policy draw ===")
    masks_by_id, random_ce_by_id = collect_masks_and_ce(
        model, batches, dev, "random", pass_easiness=False
    )

    layer_mask_rows, head_mask_rows = mask_statistics(masks_by_id, model)
    write_csv(out / "mask_layer_summary.csv", layer_mask_rows)
    write_csv(out / "mask_head_summary.csv", head_mask_rows)

    print("\n=== Random sparse vs SAME-BANK dense CE ===")
    ce_rows = random32_ce_modes(
        model, batches, dev, random_trials=args.random_trials
    )
    write_csv(out / "ce_modes.csv", ce_rows)
    for r in ce_rows:
        print(f"{r['mode']:26s} CE={r['ce']:.6f} std={r['std']:.6f}")

    print("\n=== Potential/executed geometry ===")
    geometry_rows, ratio_rows = geometry_analysis(
        model,
        module,
        batches,
        dev,
        variant="random",
        pass_easiness=False,
        layers=parse_layers(args.layers),
        max_batches=args.geometry_batches,
        sample_tokens=args.sample_tokens,
    )
    write_csv(out / "geometry_summary.csv", geometry_rows)
    write_csv(out / "executed_context_pr_ratio.csv", ratio_rows)

    spec_rows = []
    if not args.skip_specialization:
        print("\n=== Random A_h / I_h negative control ===")
        _, spec_rows = common_coalition_specialization(
            model,
            examples,
            batches,
            dev,
            masks_by_id,
            variant="random",
            pass_easiness=False,
            layers=parse_layers(args.specialization_layers),
            group_size=args.specialization_group_size,
            ablation_batch_size=args.ablation_batch_size,
            seed=args.seed,
        )
        write_csv(out / "specialization_Ah_Ih_negative_control.csv", spec_rows)

    # Expected activation frequency under each layer's K.
    expected_rows = []
    P = int(cfg.num_permanent_heads)
    E = int(cfg.num_attention_heads - P)
    for li, k_total in enumerate(cfg.random_total_heads_per_layer):
        expected = (int(k_total) - P) / E
        observed = [
            r["activation_frequency"]
            for r in head_mask_rows
            if int(r["layer"]) == li and int(r["head"]) >= P
        ]
        expected_rows.append(
            {
                "layer": li,
                "total_K": int(k_total),
                "expected_elastic_activation_frequency": expected,
                "observed_elastic_activation_frequency_mean": safe_mean(observed),
                "observed_elastic_activation_frequency_std_across_heads": safe_std(observed),
                "observed_minus_expected": safe_mean(observed) - expected,
            }
        )
    write_csv(out / "random_frequency_check.csv", expected_rows)

    lines = [
        "# HELM_32_random analysis",
        "",
        "## CE modes",
        "",
        "| mode | CE | std across random mask trials |",
        "|---|---:|---:|",
    ]
    for r in ce_rows:
        lines.append(f"| {r['mode']} | {r['ce']:.6f} | {r['std']:.6f} |")

    lines += [
        "",
        "The critical quantity is `forced_dense_same_bank` vs native Dense-32 under the same diagnostic protocol.",
        "If this randomly/sparsely trained bank remains much worse when all 32 heads are turned on, sparse exposure itself damages bank quality.",
        "",
        "## Random A_h/I_h negative control",
        "",
    ]

    if spec_rows:
        for li in parse_layers(args.specialization_layers):
            x = [
                r["specialization_gap_A_minus_I"]
                for r in spec_rows
                if int(r["layer"]) == li and np.isfinite(r["specialization_gap_A_minus_I"])
            ]
            lines.append(
                f"- Layer {li}: mean random A-I gap={safe_mean(x):+.6f}; "
                f"fraction positive={safe_mean([float(v>0) for v in x]):.3f}"
            )
        lines.append(
            "- Expected result: gaps fluctuate around zero because activation was randomly assigned."
        )

    (out / "summary.md").write_text("\n".join(lines))
    (out / "summary.json").write_text(
        json.dumps(
            {
                "ce_modes": ce_rows,
                "random_frequency_check": expected_rows,
            },
            indent=2,
        )
    )

    z = zip_results(out, "helm_32_random_analysis_results")
    print(f"\nDone: {z}")


if __name__ == "__main__":
    main()

Writing analyze_helm_32_random.py


In [4]:
%%writefile helm_analysis_core.py
"""
Shared analysis utilities for HELM routed-head experiments.

IMPORTANT NUMPY CONVERSION RULE
-------------------------------
Every Gram matrix is converted in exactly this order:

    tensor.detach().to(torch.float32).cpu().numpy().astype(np.float64)

Do not reorder that conversion.

The three companion scripts intentionally use the same:
- deterministic validation examples
- MLM masking
- CE implementation
- context/residual head geometry
- raw/directional effective-rank calculations

so their outputs are directly comparable.
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import importlib.util
import json
import math
import os
import random
import shutil
import sys
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError("pyarrow is required") from exc

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError("huggingface_hub is required") from exc


DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"
CHECKPOINT_FILE = "checkpoint-006500.pt"
TRAINING_STATE_FILE = "training_state.json"


# ---------------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------------

def get_hf_token():
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def rankdata_np(x):
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=np.float64)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        ranks[order[i:j]] = 0.5 * (i + j - 1)
        i = j
    return ranks


def spearman_np(x, y):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(rankdata_np(x), rankdata_np(y))[0, 1])


def safe_mean(x):
    a = np.asarray(list(x), dtype=np.float64)
    a = a[np.isfinite(a)]
    return float(a.mean()) if len(a) else float("nan")


def safe_std(x):
    a = np.asarray(list(x), dtype=np.float64)
    a = a[np.isfinite(a)]
    return float(a.std(ddof=1)) if len(a) > 1 else float("nan")


def safe_sem(x):
    a = np.asarray(list(x), dtype=np.float64)
    a = a[np.isfinite(a)]
    return float(a.std(ddof=1) / math.sqrt(len(a))) if len(a) > 1 else float("nan")


def gini_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, 0.0, None)
    if x.sum() <= 0 or len(x) == 0:
        return 0.0
    sx = np.sort(x)
    n = len(sx)
    return float(
        (2.0 * np.sum((np.arange(1, n + 1)) * sx) / (n * sx.sum()))
        - (n + 1) / n
    )


def effective_rank_from_gram(gram):
    gram = np.asarray(gram, dtype=np.float64)
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)
    total = vals.sum()
    if total <= 1e-18:
        return 0.0, 0.0, vals
    p = vals / total
    pp = p[p > 1e-15]
    erank = float(np.exp(-(pp * np.log(pp)).sum()))
    prank = float((total * total) / (np.square(vals).sum() + 1e-18))
    return erank, prank, vals


def directional_gram(raw_gram):
    raw_gram = np.asarray(raw_gram, dtype=np.float64)
    diag = np.clip(np.diag(raw_gram), 1e-18, None)
    denom = np.sqrt(np.outer(diag, diag))
    corr = raw_gram / denom
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def mean_abs_offdiag(mat):
    mat = np.asarray(mat, dtype=np.float64)
    if mat.shape[0] <= 1:
        return 0.0
    mask = ~np.eye(mat.shape[0], dtype=bool)
    return float(np.abs(mat[mask]).mean())


def parse_layers(text):
    return [int(x.strip()) for x in text.split(",") if x.strip()]


def write_csv(path: Path, rows: List[dict]):
    if not rows:
        path.write_text("")
        return
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def strip_state_prefixes(state):
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def import_model_file(path: Path, module_name="analysis_arch"):
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import {path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod


# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------

@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast("cuda", dtype=torch.bfloat16)
        if self.kind == "xla":
            return torch.autocast("xla", dtype=torch.bfloat16)
        return contextlib.nullcontext()


def resolve_device(requested):
    requested = requested.lower()
    if requested == "auto":
        if torch.cuda.is_available():
            requested = "cuda"
        else:
            try:
                import torch_xla.core.xla_model as xm
                return DeviceContext(xm.xla_device(), "xla", xm)
            except Exception:
                requested = "cpu"

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable")
        return DeviceContext(torch.device("cuda"), "cuda")

    if requested == "xla":
        import torch_xla.core.xla_model as xm
        return DeviceContext(xm.xla_device(), "xla", xm)

    if requested == "cpu":
        return DeviceContext(torch.device("cpu"), "cpu")

    raise ValueError(requested)


# ---------------------------------------------------------------------------
# Assets/model/data
# ---------------------------------------------------------------------------

def download_file(repo, filename, repo_type, cache_dir, token, label):
    print(f"Downloading {label}: {repo}/{filename}")
    return Path(
        hf_hub_download(
            repo_id=repo,
            filename=filename,
            repo_type=repo_type,
            token=token,
            local_dir=str(cache_dir / label),
        )
    )


def resolve_checkpoint(local_path, repo, filename, cache_dir, token):
    if local_path:
        p = Path(local_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    return download_file(repo, filename, "model", cache_dir, token, "checkpoint")


def resolve_validation(local_path, repo, filename, cache_dir, token):
    if local_path:
        p = Path(local_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    return download_file(repo, filename, "dataset", cache_dir, token, "validation")


def maybe_training_state(repo, cache_dir, token):
    try:
        return download_file(
            repo, TRAINING_STATE_FILE, "model", cache_dir, token, "training_state"
        )
    except Exception as exc:
        print(f"WARNING: training_state.json unavailable: {exc}")
        return None


def read_breakpoints(path):
    if path is None:
        return None
    try:
        state = json.loads(Path(path).read_text())
        ed = state.get("easiness_dict")
        if isinstance(ed, dict) and ed.get("breakpoints"):
            return [float(x) for x in ed["breakpoints"]]
    except Exception:
        pass
    return None


def instantiate_config(module, breakpoints=None):
    kwargs = {}
    if breakpoints is not None:
        kwargs["easiness_cdf_breakpoints"] = breakpoints
    try:
        return module.HELMConfig(**kwargs)
    except TypeError:
        return module.HELMConfig()


def load_model(module, checkpoint, dev, breakpoints=None):
    cfg = instantiate_config(module, breakpoints)
    model = module.HELMForMaskedLM(cfg)
    payload = torch.load(str(checkpoint), map_location="cpu")
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    state = strip_state_prefixes(state)

    incompat = model.load_state_dict(state, strict=False)
    missing = list(incompat.missing_keys)
    unexpected = list(incompat.unexpected_keys)
    if missing or unexpected:
        print(f"State load: {len(missing)} missing, {len(unexpected)} unexpected")
        if missing:
            print(" missing:", missing[:8])
        if unexpected:
            print(" unexpected:", unexpected[:8])
        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError("Large checkpoint/architecture mismatch")

    model.to(dev.device)
    model.eval()
    if hasattr(model, "enable_efficient_inference"):
        try:
            model.enable_efficient_inference("dense", compile=False)
        except Exception:
            pass
    return model, cfg


def deterministic_span_mask(
    ids,
    config,
    seed,
    probability=0.30,
    span_length=3,
):
    ids = ids.clone().long()
    labels = torch.full_like(ids, -100)
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))

    special = {
        int(config.bos_token_id),
        int(config.eos_token_id),
        int(config.pad_token_id),
        int(config.mask_token_id),
        int(config.unk_token_id),
    }
    candidates = [i for i, t in enumerate(ids.tolist()) if int(t) not in special]
    if not candidates:
        return ids, labels

    target = max(1, int(round(probability * len(candidates))))
    cset = set(candidates)
    perm = torch.randperm(len(candidates), generator=g).tolist()
    chosen = set()

    for pi in perm:
        if len(chosen) >= target:
            break
        st = candidates[pi]
        for p in range(st, min(st + span_length, ids.numel())):
            if p in cset:
                chosen.add(p)
                if len(chosen) >= target:
                    break

    pos = torch.tensor(sorted(chosen), dtype=torch.long)
    labels[pos] = ids[pos]
    r = torch.rand(len(pos), generator=g)

    mask_sel = r < 0.80
    rand_sel = (r >= 0.80) & (r < 0.90)
    ids[pos[mask_sel]] = int(config.mask_token_id)

    if rand_sel.any():
        ids[pos[rand_sel]] = torch.randint(
            0,
            int(config.vocab_size),
            (int(rand_sel.sum()),),
            generator=g,
        )
    return ids, labels


def prepare_examples(validation_path, config, num_examples, seq_len, seed):
    table = pq.read_table(
        str(validation_path), columns=["input_ids", "easiness_score"]
    )
    n = min(int(num_examples), table.num_rows)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(table.num_rows)[:n]

    input_col = table.column("input_ids")
    easy_col = table.column("easiness_score")
    examples = []

    for i, row_idx in enumerate(indices.tolist()):
        ids = torch.tensor(input_col[row_idx].as_py(), dtype=torch.long)[:seq_len]
        if ids.numel() < seq_len:
            ids = torch.cat(
                [
                    ids,
                    torch.full(
                        (seq_len - ids.numel(),),
                        int(config.pad_token_id),
                        dtype=torch.long,
                    ),
                ]
            )

        masked, labels = deterministic_span_mask(
            ids, config, seed=seed + 100003 * i
        )
        examples.append(
            {
                "input_ids": masked,
                "labels": labels,
                "attention_mask": (masked != int(config.pad_token_id)).long(),
                "easiness_score": torch.tensor(
                    float(easy_col[row_idx].as_py()), dtype=torch.float32
                ),
                "example_id": torch.tensor(i, dtype=torch.long),
            }
        )
    return examples


def make_batches(examples, batch_size):
    usable = (len(examples) // batch_size) * batch_size
    examples = examples[:usable]
    if not examples:
        raise RuntimeError("Not enough complete examples for one batch")

    batches = []
    for s in range(0, usable, batch_size):
        chunk = examples[s : s + batch_size]
        batches.append(
            {
                k: torch.stack([x[k] for x in chunk], dim=0)
                for k in chunk[0]
            }
        )
    return examples, batches


def make_subbatches(examples, indices, batch_size):
    idx = list(indices)
    usable = (len(idx) // batch_size) * batch_size
    idx = idx[:usable]
    out = []
    for s in range(0, usable, batch_size):
        chunk = [examples[i] for i in idx[s : s + batch_size]]
        out.append(
            {
                k: torch.stack([x[k] for x in chunk], dim=0)
                for k in chunk[0]
            }
        )
    return out


def move_batch(batch, dev):
    return {k: v.to(dev.device) for k, v in batch.items()}


# ---------------------------------------------------------------------------
# Forward / CE
# ---------------------------------------------------------------------------

def call_model(model, batch, pass_easiness=True, current_step=6500):
    kwargs = {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
    }
    if pass_easiness:
        kwargs["easiness_score"] = batch.get("easiness_score")
    else:
        kwargs["easiness_score"] = None
    kwargs["current_step"] = current_step

    try:
        out = model(**kwargs)
    except TypeError:
        kwargs.pop("current_step", None)
        try:
            out = model(**kwargs)
        except TypeError:
            kwargs.pop("easiness_score", None)
            out = model(**kwargs)

    return out[0] if isinstance(out, (tuple, list)) else out


def per_example_ce(logits, labels, chunk_tokens=128):
    B, S, V = logits.shape
    sums = torch.zeros(B, device=logits.device, dtype=torch.float32)
    counts = torch.zeros(B, device=logits.device, dtype=torch.float32)

    for s in range(0, S, chunk_tokens):
        e = min(s + chunk_tokens, S)
        lgt = logits[:, s:e, :].to(torch.float32)
        lab = labels[:, s:e]
        losses = F.cross_entropy(
            lgt.reshape(-1, V),
            lab.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, e - s)
        valid = (lab != -100).float()
        sums += (losses * valid).sum(dim=1)
        counts += valid.sum(dim=1)

    return sums / counts.clamp_min(1.0)


def evaluate_ce(
    model,
    batches,
    dev,
    pass_easiness=True,
    return_per_example=False,
):
    all_vals = {}
    total_sum = 0.0
    total_n = 0

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                logits = call_model(
                    model, batch, pass_easiness=pass_easiness
                )
                vals = per_example_ce(logits, batch["labels"])
            dev.mark_step()

            vals_cpu = vals.detach().to(torch.float32).cpu().tolist()
            ids = cpu_batch["example_id"].tolist()
            for eid, val in zip(ids, vals_cpu):
                all_vals[int(eid)] = float(val)
                total_sum += float(val)
                total_n += 1

    mean = total_sum / max(1, total_n)
    return (mean, all_vals) if return_per_example else mean


# ---------------------------------------------------------------------------
# Mask adapters
# ---------------------------------------------------------------------------

def full_mask_from_model(model, variant):
    """Return [B,L,H] hard-forward masks from the most recent forward."""
    layers = []

    if variant in {"learned", "topk"}:
        P = int(model.config.num_permanent_heads)
        for block in model.model.blocks:
            elastic = (
                block.mlt_vw_rtr.save_hard_mask.detach()
                .to(torch.float32)
                .cpu()
            )
            perm = torch.ones(
                elastic.size(0), P, dtype=torch.float32
            )
            layers.append(torch.cat([perm, elastic], dim=-1))

    elif variant == "random":
        for block in model.model.blocks:
            m = (
                block.attn.save_random_mask.detach()
                .to(torch.float32)
                .cpu()
            )
            layers.append(m)

    else:
        raise ValueError(variant)

    return torch.stack(layers, dim=1)


def collect_masks_and_ce(
    model,
    batches,
    dev,
    variant,
    pass_easiness,
):
    masks = {}
    ce_by_id = {}

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                logits = call_model(
                    model, batch, pass_easiness=pass_easiness
                )
                vals = per_example_ce(logits, batch["labels"])
            dev.mark_step()

            fm = full_mask_from_model(model, variant)
            ids = cpu_batch["example_id"].tolist()
            vals = vals.detach().to(torch.float32).cpu().tolist()

            for bi, eid in enumerate(ids):
                masks[int(eid)] = fm[bi].numpy().astype(np.float32)
                ce_by_id[int(eid)] = float(vals[bi])

    return masks, ce_by_id


class LearnedRouterOverride:
    """
    Forward-hook replacement for HELM_7c/7e/constant-K router outputs.

    mode:
      dense
      permanent
      random_same_count
      dense_minus (requires target layer/head)
      forced (requires dict layer->[B,H] full masks)
    """

    def __init__(
        self,
        model,
        mode,
        target_layer=None,
        target_head=None,
        forced_masks=None,
    ):
        self.model = model
        self.mode = mode
        self.target_layer = target_layer
        self.target_head = target_head
        self.forced_masks = forced_masks or {}
        self.handles = []

    def _hook(self, li):
        def hook(module, inputs, output):
            B, H, _, _ = output.shape
            P = int(self.model.config.num_permanent_heads)

            if self.mode == "dense":
                return torch.ones_like(output)

            if self.mode == "permanent":
                result = torch.zeros_like(output)
                result[:, :P, :, :] = 1
                return result

            if self.mode == "dense_minus":
                result = torch.ones_like(output)
                if li == self.target_layer:
                    result[:, int(self.target_head), :, :] = 0
                return result

            if self.mode == "random_same_count":
                # output forward values are hard 0/1 even though an STE exists.
                k_elastic = (
                    output[:, P:, 0, 0].detach().float().sum(dim=-1).long()
                )
                E = H - P
                scores = torch.rand(
                    B, E, device=output.device, dtype=torch.float32
                )
                # Random unique ranks 0..E-1 per example.
                order = torch.argsort(scores, dim=-1, descending=True)
                ranks = torch.argsort(order, dim=-1)
                elastic = (
                    ranks < k_elastic.view(B, 1)
                ).to(output.dtype)
                perm = torch.ones(
                    B, P, device=output.device, dtype=output.dtype
                )
                full = torch.cat([perm, elastic], dim=-1)
                return full.view(B, H, 1, 1)

            if self.mode == "forced":
                m = self.forced_masks[li].to(
                    device=output.device, dtype=output.dtype
                )
                return m.view(B, H, 1, 1)

            raise ValueError(self.mode)

        return hook

    def __enter__(self):
        for li, block in enumerate(self.model.model.blocks):
            self.handles.append(
                block.mlt_vw_rtr.register_forward_hook(self._hook(li))
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


class RandomAttentionMaskOverride:
    """
    Override HELM_32_random._random_full_mask for controlled CE tests.

    mode:
      dense
      permanent
      dense_minus
      forced
    """

    def __init__(
        self,
        model,
        mode,
        target_layer=None,
        target_head=None,
        forced_masks=None,
    ):
        self.model = model
        self.mode = mode
        self.target_layer = target_layer
        self.target_head = target_head
        self.forced_masks = forced_masks or {}
        self.originals = []
        self.old_modes = []

    def __enter__(self):
        P = int(self.model.config.num_permanent_heads)
        H = int(self.model.config.num_attention_heads)

        for li, block in enumerate(self.model.model.blocks):
            attn = block.attn
            self.old_modes.append(attn.routing_mode)
            attn.set_routing_mode("random")
            original = attn._random_full_mask
            self.originals.append(original)

            def make_override(layer_idx, attn_obj):
                def fn(_self, batch_size, device, dtype):
                    if self.mode == "dense":
                        return torch.ones(
                            batch_size, H, device=device, dtype=dtype
                        )

                    if self.mode == "permanent":
                        result = torch.zeros(
                            batch_size, H, device=device, dtype=dtype
                        )
                        result[:, :P] = 1
                        return result

                    if self.mode == "dense_minus":
                        result = torch.ones(
                            batch_size, H, device=device, dtype=dtype
                        )
                        if layer_idx == self.target_layer:
                            result[:, int(self.target_head)] = 0
                        return result

                    if self.mode == "forced":
                        return self.forced_masks[layer_idx].to(
                            device=device, dtype=dtype
                        )

                    raise ValueError(self.mode)
                return types.MethodType(fn, attn_obj)

            attn._random_full_mask = make_override(li, attn)

        return self

    def __exit__(self, exc_type, exc, tb):
        for block, original, old_mode in zip(
            self.model.model.blocks, self.originals, self.old_modes
        ):
            block.attn._random_full_mask = original
            block.attn.set_routing_mode(old_mode)


def forced_masks_for_ids(masks_by_id, ids, dev, target_layer, target_head, value):
    L, H = next(iter(masks_by_id.values())).shape
    out = {}
    for li in range(L):
        arr = np.stack([masks_by_id[int(i)][li] for i in ids], axis=0)
        if li == target_layer:
            arr[:, target_head] = float(value)
        out[li] = torch.tensor(arr, dtype=torch.float32, device=dev.device)
    return out


# ---------------------------------------------------------------------------
# Attention geometry
# ---------------------------------------------------------------------------

class AttentionInputCapture:
    def __init__(self, model, layers):
        self.model = model
        self.layers = list(layers)
        self.data = {}
        self.handles = []

    def _hook(self, li):
        def hook(module, inputs):
            # All current models have hidden_states and attention_mask first.
            self.data[li] = (inputs[0].detach(), inputs[1].detach())
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(
                    self._hook(li)
                )
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def unmasked_attention_context(module, attn, hidden_states, attention_mask):
    qkv_proj = module.cast_linear(hidden_states, attn.qkv)
    B, S, _ = hidden_states.shape

    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)
    q = q.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    k = k.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    v = v.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)

    q = module.justnorm(q)
    k = module.justnorm(k)
    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = attn.sqk * (
        attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale
    )
    sqk = sqk.view(
        1, attn.num_attention_heads, 1, attn.d_head
    ).to(q.dtype)

    q = sqk * q
    k = sqk * k

    context = F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )

    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (
            context * vn
        ).sum(dim=-1, keepdim=True) * vn

    return context


def add_gram(acc, key, value):
    if key not in acc:
        acc[key] = np.zeros_like(value, dtype=np.float64)
    acc[key] += value


def geometry_analysis(
    model,
    module,
    batches,
    dev,
    variant,
    pass_easiness,
    layers,
    max_batches,
    sample_tokens,
):
    """
    Produce potential and executed context/residual Gram matrices.

    IMPORTANT:
      Conversion is exactly:
      detach -> float32 -> cpu -> numpy -> float64
    """
    grams = {}
    pr_ratio_examples = {li: [] for li in layers}

    with torch.no_grad():
        for cpu_batch in batches[:max_batches]:
            batch = move_batch(cpu_batch, dev)

            with AttentionInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = call_model(
                        model, batch, pass_easiness=pass_easiness
                    )
                dev.mark_step()

            full_masks = full_mask_from_model(model, variant).to(dev.device)

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn

                with dev.autocast():
                    context = unmasked_attention_context(
                        module, attn, hidden, attn_mask
                    )  # [B,H,S,d]

                    S = context.size(2)
                    T = min(int(sample_tokens), S)
                    positions = torch.linspace(
                        0, S - 1, steps=T, device=context.device
                    ).long()
                    c = context.index_select(2, positions)

                    mask = full_masks[:, li, :].to(
                        device=c.device, dtype=c.dtype
                    ).view(c.size(0), c.size(1), 1, 1)

                    c_exec = c * mask

                    W = attn.output.weight.to(c.dtype).view(
                        attn.hidden_size,
                        attn.num_attention_heads,
                        attn.d_head,
                    )

                    y = torch.einsum("bhtd,ohd->bhto", c, W)
                    y_exec = y * mask

                    # Aggregated Gram matrices.
                    for state, c_use, y_use in (
                        ("potential", c, y),
                        ("executed", c_exec, y_exec),
                    ):
                        cflat = (
                            c_use.permute(1, 0, 2, 3)
                            .contiguous()
                            .view(c_use.size(1), -1)
                        )
                        yflat = (
                            y_use.permute(1, 0, 2, 3)
                            .contiguous()
                            .view(y_use.size(1), -1)
                        )

                        cgram = cflat @ cflat.T
                        ygram = yflat @ yflat.T

                        # REQUIRED CONVERSION ORDER.
                        cgram_np = (
                            cgram.detach()
                            .to(torch.float32)
                            .cpu()
                            .numpy()
                            .astype(np.float64)
                        )
                        ygram_np = (
                            ygram.detach()
                            .to(torch.float32)
                            .cpu()
                            .numpy()
                            .astype(np.float64)
                        )

                        add_gram(grams, (li, state, "context"), cgram_np)
                        add_gram(grams, (li, state, "residual"), ygram_np)

                    # Exact per-example context participation-rank ratio r_PR / K.
                    flat = c_exec.to(torch.float32).reshape(
                        c_exec.size(0), c_exec.size(1), -1
                    )
                    flat = flat / math.sqrt(float(max(1, flat.size(-1))))
                    g = torch.bmm(flat, flat.transpose(1, 2))
                    tr = torch.diagonal(g, dim1=-2, dim2=-1).sum(-1)
                    tr2 = g.square().sum(dim=(-2, -1))
                    rpr = tr.square() / (tr2 + 1e-12)
                    K = full_masks[:, li, :].sum(-1).clamp_min(1).to(rpr.device)
                    ratio = rpr / K
                    pr_ratio_examples[li].extend(
                        ratio.detach().to(torch.float32).cpu().tolist()
                    )

    rows = []

    for key, gram in sorted(grams.items()):
        li, state, space = key
        raw_er, raw_pr, _ = effective_rank_from_gram(gram)
        dgram = directional_gram(gram)
        dir_er, dir_pr, _ = effective_rank_from_gram(dgram)

        energy = np.clip(np.diag(gram), 0.0, None)
        ef = energy / max(1e-18, energy.sum())
        top_sorted = np.sort(ef)[::-1]

        rows.append(
            {
                "layer": li,
                "state": state,
                "space": space,
                "heads": gram.shape[0],
                "raw_entropy_rank": raw_er,
                "raw_entropy_ratio": raw_er / gram.shape[0],
                "raw_participation_rank": raw_pr,
                "raw_participation_ratio": raw_pr / gram.shape[0],
                "directional_entropy_rank": dir_er,
                "directional_entropy_ratio": dir_er / gram.shape[0],
                "directional_participation_rank": dir_pr,
                "directional_participation_ratio": dir_pr / gram.shape[0],
                "mean_abs_cos": mean_abs_offdiag(dgram),
                "energy_gini": gini_np(energy),
                "top1_energy_fraction": float(top_sorted[:1].sum()),
                "top4_energy_fraction": float(top_sorted[:4].sum()),
            }
        )

    ratio_rows = []
    for li, vals in pr_ratio_examples.items():
        ratio_rows.append(
            {
                "layer": li,
                "mean_context_pr_ratio_per_example": safe_mean(vals),
                "std_context_pr_ratio_per_example": safe_std(vals),
                "min_context_pr_ratio_per_example": float(np.min(vals)) if vals else float("nan"),
                "max_context_pr_ratio_per_example": float(np.max(vals)) if vals else float("nan"),
            }
        )

    return rows, ratio_rows


# ---------------------------------------------------------------------------
# Router/mask statistics
# ---------------------------------------------------------------------------

def mask_statistics(masks_by_id, model):
    ids = sorted(masks_by_id)
    arr = np.stack([masks_by_id[i] for i in ids], axis=0)  # [N,L,H]
    N, L, H = arr.shape
    P = int(getattr(model.config, "num_permanent_heads", 0))

    layer_rows = []
    head_rows = []

    for li in range(L):
        counts = arr[:, li, :].sum(axis=-1)
        unique = np.unique(arr[:, li, :], axis=0).shape[0]

        layer_rows.append(
            {
                "layer": li,
                "mean_total_heads": float(counts.mean()),
                "std_total_heads": float(counts.std()),
                "min_total_heads": float(counts.min()),
                "max_total_heads": float(counts.max()),
                "unique_masks": int(unique),
                "unique_mask_fraction": float(unique / N),
            }
        )

        for h in range(H):
            head_rows.append(
                {
                    "layer": li,
                    "head": h,
                    "is_permanent_slot": int(h < P),
                    "activation_frequency": float(arr[:, li, h].mean()),
                }
            )

    return layer_rows, head_rows



def learned_router_calibration(
    model,
    batches,
    dev,
    pass_easiness=True,
):
    """Collect per-example, per-layer learned-router counts/targets/errors."""
    rows = []

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                _ = call_model(
                    model, batch, pass_easiness=pass_easiness
                )
            dev.mark_step()

            ids = cpu_batch["example_id"].tolist()
            easiness = cpu_batch["easiness_score"].tolist()

            for li, block in enumerate(model.model.blocks):
                router = block.mlt_vw_rtr
                actual = (
                    router.save_total_head_count.detach()
                    .to(torch.float32).cpu().tolist()
                )
                target = (
                    router.save_target_total_head_count.detach()
                    .to(torch.float32).cpu().tolist()
                )
                logits = (
                    router.save_router_logits.detach()
                    .to(torch.float32).cpu()
                )
                sig = torch.sigmoid(logits)

                for bi, eid in enumerate(ids):
                    rows.append(
                        {
                            "example_id": int(eid),
                            "layer": li,
                            "easiness_score": float(easiness[bi]),
                            "actual_total_heads": float(actual[bi]),
                            "target_total_heads": float(target[bi]),
                            "count_error": float(actual[bi] - target[bi]),
                            "mean_sigmoid": float(sig[bi].mean()),
                            "sigmoid_saturation_fraction": float(
                                ((sig[bi] < 0.05) | (sig[bi] > 0.95))
                                .float().mean()
                            ),
                            "mean_ste_derivative": float(
                                (sig[bi] * (1.0 - sig[bi])).mean()
                            ),
                        }
                    )

    return rows

# ---------------------------------------------------------------------------
# A_h / I_h common-coalition specialization
# ---------------------------------------------------------------------------

def common_coalition_specialization(
    model,
    examples,
    batches,
    dev,
    masks_by_id,
    variant,
    pass_easiness,
    layers,
    group_size,
    ablation_batch_size,
    seed,
):
    """
    For each elastic/non-permanent head h:
      A_h = examples where baseline policy selected h
      I_h = examples where baseline policy did not select h

    Evaluate BOTH groups under the SAME all-head coalition and remove h.

      utility = CE(all heads except h) - CE(all heads)

    Positive A-I gap means the policy tends to select h on examples where h has
    greater exact single-head utility under the common all-head context.

    For Random-32 this is a negative control: because A/I assignment is random
    wrt input, the expected gap is zero.
    """
    P = int(getattr(model.config, "num_permanent_heads", 0))
    H = int(model.config.num_attention_heads)
    rng = np.random.default_rng(seed)

    Override = (
        RandomAttentionMaskOverride
        if variant == "random"
        else LearnedRouterOverride
    )

    with Override(model, "dense"):
        dense_mean, dense_ce = evaluate_ce(
            model,
            batches,
            dev,
            pass_easiness=pass_easiness,
            return_per_example=True,
        )

    all_ids = sorted(masks_by_id)
    rows = []

    for li in layers:
        for h in range(P, H):
            A = [i for i in all_ids if masks_by_id[i][li, h] > 0.5]
            I = [i for i in all_ids if masks_by_id[i][li, h] <= 0.5]

            rng.shuffle(A)
            rng.shuffle(I)

            nA = min(group_size, len(A))
            nI = min(group_size, len(I))
            nA = (nA // ablation_batch_size) * ablation_batch_size
            nI = (nI // ablation_batch_size) * ablation_batch_size
            A = A[:nA]
            I = I[:nI]

            def eval_group(ids):
                if not ids:
                    return []
                bs = make_subbatches(examples, ids, ablation_batch_size)
                with Override(
                    model,
                    "dense_minus",
                    target_layer=li,
                    target_head=h,
                ):
                    _, vals = evaluate_ce(
                        model,
                        bs,
                        dev,
                        pass_easiness=pass_easiness,
                        return_per_example=True,
                    )
                return [vals[i] - dense_ce[i] for i in ids if i in vals]

            du_A = eval_group(A)
            du_I = eval_group(I)

            rows.append(
                {
                    "layer": li,
                    "head": h,
                    "activation_frequency": float(
                        np.mean([masks_by_id[i][li, h] for i in all_ids])
                    ),
                    "active_examples_used": len(du_A),
                    "inactive_examples_used": len(du_I),
                    "utility_active_mean": safe_mean(du_A),
                    "utility_active_sem": safe_sem(du_A),
                    "utility_inactive_mean": safe_mean(du_I),
                    "utility_inactive_sem": safe_sem(du_I),
                    "specialization_gap_A_minus_I": (
                        safe_mean(du_A) - safe_mean(du_I)
                        if du_A and du_I
                        else float("nan")
                    ),
                }
            )

            print(
                f"L{li:02d} h{h:02d} f="
                f"{rows[-1]['activation_frequency']:.3f} "
                f"A-I={rows[-1]['specialization_gap_A_minus_I']:+.5f}"
            )

    return dense_mean, rows


# ---------------------------------------------------------------------------
# CE mode helpers
# ---------------------------------------------------------------------------

def learned_ce_modes(
    model,
    batches,
    dev,
    pass_easiness,
    random_trials=3,
):
    rows = []

    routed = evaluate_ce(
        model, batches, dev, pass_easiness=pass_easiness
    )
    rows.append({"mode": "learned_routed", "ce": routed, "std": 0.0})

    with LearnedRouterOverride(model, "dense"):
        dense = evaluate_ce(
            model, batches, dev, pass_easiness=pass_easiness
        )
    rows.append({"mode": "forced_dense_all32", "ce": dense, "std": 0.0})

    with LearnedRouterOverride(model, "permanent"):
        perm = evaluate_ce(
            model, batches, dev, pass_easiness=pass_easiness
        )
    rows.append({"mode": "permanent_only", "ce": perm, "std": 0.0})

    vals = []
    for _ in range(int(random_trials)):
        with LearnedRouterOverride(model, "random_same_count"):
            vals.append(
                evaluate_ce(
                    model,
                    batches,
                    dev,
                    pass_easiness=pass_easiness,
                )
            )
    rows.append(
        {
            "mode": "random_same_count",
            "ce": safe_mean(vals),
            "std": safe_std(vals),
        }
    )

    return rows


def random32_ce_modes(
    model,
    batches,
    dev,
    random_trials=5,
):
    rows = []

    # Different random mask draw each trial.
    vals = []
    model.set_routing_mode("random")
    for _ in range(int(random_trials)):
        vals.append(
            evaluate_ce(model, batches, dev, pass_easiness=False)
        )
    rows.append(
        {
            "mode": "random_sparse",
            "ce": safe_mean(vals),
            "std": safe_std(vals),
        }
    )

    with RandomAttentionMaskOverride(model, "dense"):
        dense = evaluate_ce(model, batches, dev, pass_easiness=False)
    rows.append({"mode": "forced_dense_same_bank", "ce": dense, "std": 0.0})

    with RandomAttentionMaskOverride(model, "permanent"):
        perm = evaluate_ce(model, batches, dev, pass_easiness=False)
    rows.append({"mode": "permanent_only", "ce": perm, "std": 0.0})

    return rows


def zip_results(output_dir: Path, name: str):
    # ZIP MUST be outside output_dir to prevent recursive self-zipping.
    zip_base = output_dir.parent / name
    old = zip_base.with_suffix(".zip")
    if old.exists():
        old.unlink()
    return shutil.make_archive(
        str(zip_base), "zip", root_dir=output_dir
    )

Writing helm_analysis_core.py


## 4. Single-checkpoint audit

For an apples-to-apples comparison with HELM_7c, start with **checkpoint 6,500**. Exact ablation is restricted to layers 0, 5, and 11 to keep the intervention interpretable and reasonably sized.

The post-hoc head-count curve keeps 4/8/12/16 heads per layer using either the largest measured residual-energy heads or random same-count subsets. Remember: this is a pruning/robustness test, **not** a substitute for training a smaller model from scratch.


In [5]:
!python analyze_helm_32_random.py \
    --device xla \
    --batch-size 2 \
    --num-examples 64

E0000 00:00:1786579993.080563     172 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238

checkpoint-006500.pt: downloading bytes:                   |  0.00B            
checkpoint-006500.pt: downloading bytes: ██▏               |  453MB, 36.3MB/s  
checkpoint-006500.pt: downloading bytes: ████▏             |  878MB, 68.2MB/s  
checkpoint-006500.pt: downloading bytes: ████▌             |  938MB, 71.4MB/s  
checkpoint-006500.pt: downloading bytes: ███████▍          | 1.54GB,  115MB/s  
checkpoint-006500.pt: downloading bytes: ███████████████▎  | 3.17GB,  199MB/s  
checkpoint-006500.pt: downloading bytes: ████████████████▎ | 3.38GB,  144MB/s  
checkpoint-006500.pt: reconstructing file:  94%|█▉| 3.51GB / 3.72GB,  121MB/s  
checkpoint-006500.pt: downloading bytes: ██████████████████| 3.39GB, 

## 5. Rank over training

This uses fewer examples/batches because it repeats the same geometry analysis across multiple checkpoints. By default it chooses five available checkpoints evenly across the run and measures layers 0, 5, and 11.


In [6]:
# import subprocess, sys

# cmd = [
#     sys.executable, "audit_dense32_rank_over_time.py",
#     "--device", "xla",
#     "--num-checkpoints", "5",
#     "--layers", "0,5,11",
#     "--num-examples", "8",
#     "--batch-size", "2",
#     "--functional-batches", "2",
#     "--functional-sample-tokens", "16",
# ]

# print(" ".join(cmd))
# subprocess.run(cmd, check=True)


## Outputs to send back

For the single-checkpoint audit, send:

- `dense16_assumption_audit/summary.md`
- `dense16_assumption_audit_results.zip`

For the training-dynamics audit, send:

- `dense16_rank_over_time/summary.md`
- `dense16_rank_over_time_results.zip`

The most important comparisons will be **raw vs directional rank**, **context vs residual rank**, **RMS vs exact ΔCE**, and **how all of those quantities move as CE improves during training**.
